In [1]:
# ============================================================
# Section 1: Imports and Photos Library paths
# ============================================================

import osxphotos

from explorephotoslibrary import *


USE_INVENTORY_CACHE = True


PHOTOS_LIBRARY_PATHS = {
    "backup_20250317": "/Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary",
    "current_default": "/Users/huohsien/Pictures/Photos Library.photoslibrary",
    "test": "/Users/huohsien/Pictures/test.photoslibrary",
}


In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

def load_or_build_inventory(library_key):
    library_path = PHOTOS_LIBRARY_PATHS[library_key]

    if USE_INVENTORY_CACHE:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            inventory = load_inventory_cache(library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    print("=" * 80)
    print(f"Build inventory: {library_key}")
    print("=" * 80)

    osx_assets = osxphotos.PhotosDB(library_path).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Load inventory cache: backup_20250317
loaded inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.47
inventory assets: 71599
inventory albums: 5172
inventory folders: 35
movies: 6240
hidden: 0
favorites: 699
descriptions: 727
keywords: 23758

Load inventory cache: current_default
loaded inventory cache: data/inventory_cache/current_default.inventory.pkl.gz
elapsed seconds: 0.66
inventory assets: 94739
inventory albums: 5956
inventory folders: 33
movies: 7406
hidden: 1
favorites: 768
descriptions: 2293
keywords: 27529


In [3]:
# ============================================================
# Test 2: Photo Library asset unique ID preflight
# ============================================================

import time

t0 = time.perf_counter()

fill_photo_library_asset_unique_ids(inventory_backup)

t1 = time.perf_counter()

fill_photo_library_asset_unique_ids(inventory_current)

t2 = time.perf_counter()

print("fill backup unique IDs elapsed seconds:", round(t1 - t0, 3))
print("fill current unique IDs elapsed seconds:", round(t2 - t1, 3))
print("fill total elapsed seconds:", round(t2 - t0, 3))
print()

backup_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    "BACKUP Photo Library asset unique ID audit",
)

print()

current_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_current,
    "CURRENT DEFAULT Photo Library asset unique ID audit",
)

print()
print("backup_unique_id_ok:", backup_unique_id_ok)
print("current_unique_id_ok:", current_unique_id_ok)
print("ready_for_cross_library_comparison:", backup_unique_id_ok and current_unique_id_ok)

fill backup unique IDs elapsed seconds: 64.796
fill current unique IDs elapsed seconds: 70.123
fill total elapsed seconds: 134.92

BACKUP Photo Library asset unique ID audit
--------------------------------------------------------------------------------
total asset count: 71599
generated unique ID count: 71566
assets without unique ID: 0
duplicate unique ID group count: 33
duplicate asset count: 66
is Photo Library asset unique ID scheme unique: False

Duplicate unique ID groups
--------------------------------------------------------------------------------
photo_library_asset_unique_id: ('IMG_0234.PNG', '11-19 00:23:16.639491', 408582, None)
asset count: 2
  uuid: 5E51F8F4-6775-4851-A5C4-B297BE1597B5
  filename: 5E51F8F4-6775-4851-A5C4-B297BE1597B5.png
  original_filename: IMG_0234.PNG
  date: 2020-11-19T00:23:16.639491+08:00
  date_added: 2020-11-19T00:23:29.247762+08:00
  path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS 

In [4]:
# ============================================================
# TEMP: Diagnose potential duplicate groups by SHA256
# ============================================================

import hashlib
import os
import time


def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
    path = asset.get("path")

    if path is None:
        return None

    if not os.path.exists(path):
        return None

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def build_potential_duplicate_groups_by_unique_id(inventory):
    unique_id_to_assets = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            continue

        if unique_id not in unique_id_to_assets:
            unique_id_to_assets[unique_id] = []

        unique_id_to_assets[unique_id].append(asset)

    return {
        unique_id: assets
        for unique_id, assets in unique_id_to_assets.items()
        if len(assets) > 1
    }


def diagnose_potential_duplicate_groups_with_sha256(
    inventory,
    label,
    max_true_duplicate_groups_to_print=20,
    max_key_collision_groups_to_print=20,
):
    start_time = time.perf_counter()

    potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

    true_content_duplicate_groups = []
    key_collision_groups = []
    sha_error_assets = []

    checked_asset_count = 0

    for unique_id, assets in potential_groups.items():
        sha256_to_assets = {}

        for asset in assets:
            checked_asset_count += 1
            sha256 = compute_sha256_for_asset(asset)

            if sha256 is None:
                sha_error_assets.append(asset)
                continue

            if sha256 not in sha256_to_assets:
                sha256_to_assets[sha256] = []

            sha256_to_assets[sha256].append(asset)

        duplicate_sha_groups = {
            sha256: sha_assets
            for sha256, sha_assets in sha256_to_assets.items()
            if len(sha_assets) > 1
        }

        if duplicate_sha_groups:
            for sha256, sha_assets in duplicate_sha_groups.items():
                true_content_duplicate_groups.append(
                    {
                        "unique_id": unique_id,
                        "sha256": sha256,
                        "assets": sha_assets,
                    }
                )

        if len(sha256_to_assets) > 1:
            key_collision_groups.append(
                {
                    "unique_id": unique_id,
                    "sha256_to_assets": sha256_to_assets,
                }
            )

    elapsed = time.perf_counter() - start_time

    print(label)
    print("-" * 80)
    print("potential duplicate unique_id group count:", len(potential_groups))
    print("checked asset count:", checked_asset_count)
    print("sha error asset count:", len(sha_error_assets))
    print("true content duplicate group count:", len(true_content_duplicate_groups))
    print("key collision group count:", len(key_collision_groups))
    print("elapsed seconds:", round(elapsed, 3))

    print()
    print("TRUE CONTENT DUPLICATE GROUPS")
    print("-" * 80)

    for index, group in enumerate(true_content_duplicate_groups):
        if index >= max_true_duplicate_groups_to_print:
            print("... more true content duplicate groups not printed")
            break

        print("unique_id:", group["unique_id"])
        print("sha256:", group["sha256"])
        print("asset count:", len(group["assets"]))

        for asset in group["assets"]:
            print("  uuid:", asset.get("uuid"))
            print("  original_filename:", asset.get("original_filename"))
            print("  filename:", asset.get("filename"))
            print("  date:", asset.get("date"))
            print("  date_added:", asset.get("date_added"))
            print("  file_size_bytes:", asset.get("file_size_bytes"))
            print("  path:", asset.get("path"))

        print("-" * 80)

    print()
    print("KEY COLLISION GROUPS")
    print("-" * 80)

    for index, group in enumerate(key_collision_groups):
        if index >= max_key_collision_groups_to_print:
            print("... more key collision groups not printed")
            break

        print("unique_id:", group["unique_id"])
        print("sha256 count:", len(group["sha256_to_assets"]))

        for sha256, assets in group["sha256_to_assets"].items():
            print("  sha256:", sha256)
            print("  asset count:", len(assets))

            for asset in assets:
                print("    uuid:", asset.get("uuid"))
                print("    original_filename:", asset.get("original_filename"))
                print("    filename:", asset.get("filename"))
                print("    date:", asset.get("date"))
                print("    date_added:", asset.get("date_added"))
                print("    file_size_bytes:", asset.get("file_size_bytes"))
                print("    path:", asset.get("path"))

        print("-" * 80)

    return {
        "potential_groups": potential_groups,
        "true_content_duplicate_groups": true_content_duplicate_groups,
        "key_collision_groups": key_collision_groups,
        "sha_error_assets": sha_error_assets,
    }


backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256(
    inventory_backup,
    "BACKUP potential duplicate diagnostic",
)

print()

current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256(
    inventory_current,
    "CURRENT potential duplicate diagnostic",
)

BACKUP potential duplicate diagnostic
--------------------------------------------------------------------------------
potential duplicate unique_id group count: 33
checked asset count: 66
sha error asset count: 0
true content duplicate group count: 33
key collision group count: 0
elapsed seconds: 52.961

TRUE CONTENT DUPLICATE GROUPS
--------------------------------------------------------------------------------
unique_id: ('IMG_0234.PNG', '11-19 00:23:16.639491', 408582, None)
sha256: 9c711a04be7bac97c599c0c3cb45ad4301f0679b66fa364402277964a9fda1d4
asset count: 2
  uuid: 5E51F8F4-6775-4851-A5C4-B297BE1597B5
  original_filename: IMG_0234.PNG
  filename: 5E51F8F4-6775-4851-A5C4-B297BE1597B5.png
  date: 2020-11-19T00:23:16.639491+08:00
  date_added: 2020-11-19T00:23:29.247762+08:00
  file_size_bytes: 408582
  path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/5/5E51F8F4-6775-4851-A5C4

In [5]:
# ============================================================
# Test 2: Run inventory comparison summary
# ============================================================

if not (backup_unique_id_ok and current_unique_id_ok):
    raise RuntimeError(
        "Photo Library asset unique ID preflight failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
)

summarize_diff_records(diff_records)


RuntimeError: Photo Library asset unique ID preflight failed. Do not run cross-library comparison yet.

In [ ]:
# ============================================================
# Test 2: Inspect diff records by change type
# ============================================================

target_change_type = ChangeType.ASSET_MISSING_FROM_CURRENT.value

matched_records = [
    record
    for record in diff_records
    if record["change_type"] == target_change_type
]

print("target change type:", target_change_type)
print("matched record count:", len(matched_records))
print()

for record in matched_records[:20]:
    backup_asset = record["backup_object"]
    current_asset = record["current_object"]

    print("change_type:", record["change_type"])
    print("description:", record["change_type_description"])
    print("changed_field:", record["changed_field"])
    print("backup_value:", record["backup_value"])
    print("current_value:", record["current_value"])

    if backup_asset is not None:
        print("backup uuid:", backup_asset["uuid"])
        print("backup original_filename:", backup_asset["original_filename"])
        print("backup date:", backup_asset["date"])

    if current_asset is not None:
        print("current uuid:", current_asset["uuid"])
        print("current original_filename:", current_asset["original_filename"])
        print("current date:", current_asset["date"])

    print("-" * 80)


In [ ]:
# ============================================================
# Test 2: Inventory comparison helpers
# ============================================================

from enum import Enum


class ChangeType(str, Enum):
    # Asset existence
    ASSET_MISSING_FROM_CURRENT = "ASSET_MISSING_FROM_CURRENT"
    ASSET_NEW_IN_CURRENT = "ASSET_NEW_IN_CURRENT"

    # Asset metadata fields
    ASSET_FIELD_CHANGED_DESCRIPTION = "ASSET_FIELD_CHANGED__description"
    ASSET_FIELD_CHANGED_KEYWORDS = "ASSET_FIELD_CHANGED__keywords"
    ASSET_FIELD_CHANGED_FAVORITE = "ASSET_FIELD_CHANGED__favorite"
    ASSET_FIELD_CHANGED_HIDDEN = "ASSET_FIELD_CHANGED__hidden"
    ASSET_FIELD_CHANGED_DATE = "ASSET_FIELD_CHANGED__date"
    ASSET_FIELD_CHANGED_DATE_ADDED = "ASSET_FIELD_CHANGED__date_added"
    ASSET_FIELD_CHANGED_ORIGINAL_FILENAME = "ASSET_FIELD_CHANGED__original_filename"
    ASSET_FIELD_CHANGED_IS_MOVIE = "ASSET_FIELD_CHANGED__is_movie"

    # Asset relationship metadata
    ASSET_ALBUM_MEMBERSHIP_REMOVED = "ASSET_ALBUM_MEMBERSHIP_REMOVED"
    ASSET_ALBUM_MEMBERSHIP_ADDED = "ASSET_ALBUM_MEMBERSHIP_ADDED"

    ASSET_FOLDER_PATHS_REMOVED = "ASSET_FOLDER_PATHS_REMOVED"
    ASSET_FOLDER_PATHS_ADDED = "ASSET_FOLDER_PATHS_ADDED"
    ASSET_FOLDER_PATHS_CHANGED = "ASSET_FOLDER_PATHS_CHANGED"

    # Album existence and folder relationship
    ALBUM_MISSING_FROM_CURRENT = "ALBUM_MISSING_FROM_CURRENT"
    ALBUM_NEW_IN_CURRENT = "ALBUM_NEW_IN_CURRENT"

    ALBUM_FOLDER_PATHS_REMOVED = "ALBUM_FOLDER_PATHS_REMOVED"
    ALBUM_FOLDER_PATHS_ADDED = "ALBUM_FOLDER_PATHS_ADDED"
    ALBUM_FOLDER_PATHS_CHANGED = "ALBUM_FOLDER_PATHS_CHANGED"

    ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS = "ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS"

    # Folder path existence
    FOLDER_PATH_MISSING_FROM_CURRENT = "FOLDER_PATH_MISSING_FROM_CURRENT"
    FOLDER_PATH_NEW_IN_CURRENT = "FOLDER_PATH_NEW_IN_CURRENT"


CHANGE_TYPE_DESCRIPTIONS = {
    ChangeType.ASSET_MISSING_FROM_CURRENT:
        "Asset exists in backup but not in the current default Photos Library. This is high-priority because it may mean the photo or video disappeared from the live iCloud library.",

    ChangeType.ASSET_NEW_IN_CURRENT:
        "Asset exists in the current default Photos Library but not in backup. Usually normal because current_default is later than the 2025-03-17 backup.",

    ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION:
        "Same derived asset ID exists in both libraries, but the caption/description changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_KEYWORDS:
        "Same derived asset ID exists in both libraries, but keyword metadata changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_FAVORITE:
        "Same derived asset ID exists in both libraries, but favorite status changed. This may be real user action or metadata loss.",

    ChangeType.ASSET_FIELD_CHANGED_HIDDEN:
        "Same derived asset ID exists in both libraries, but hidden status changed. This may be real user action or metadata difference.",

    ChangeType.ASSET_FIELD_CHANGED_DATE:
        "Same derived asset ID exists in both libraries, but asset date changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED:
        "Same derived asset ID exists in both libraries, but date_added changed. This may be less important because import/sync timing can differ.",

    ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME:
        "Same derived asset ID exists in both libraries, but original filename changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE:
        "Same derived asset ID exists in both libraries, but is_movie changed. This is highly unusual.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED:
        "Asset still exists in current, but one or more album memberships from backup are missing.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED:
        "Asset has album memberships in current that did not exist in backup. Often normal because current is later.",

    ChangeType.ASSET_FOLDER_PATHS_REMOVED:
        "Asset still exists in current, but backup folder-path relationships are gone in the current default Photos Library.",

    ChangeType.ASSET_FOLDER_PATHS_ADDED:
        "Asset has folder-path relationships in current that did not exist in backup. Often normal or caused by later organization.",

    ChangeType.ASSET_FOLDER_PATHS_CHANGED:
        "Asset still exists in both libraries, but folder-path relationships changed.",

    ChangeType.ALBUM_MISSING_FROM_CURRENT:
        "Album title exists in backup but not in the current default Photos Library. This may require album reconstruction.",

    ChangeType.ALBUM_NEW_IN_CURRENT:
        "Album title exists in the current default Photos Library but not in backup. Usually normal because current is later.",

    ChangeType.ALBUM_FOLDER_PATHS_REMOVED:
        "Album still exists in current, but it is no longer inside the folder path recorded in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_ADDED:
        "Album gained folder-path relationships in current that did not exist in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_CHANGED:
        "Album still exists in both libraries, but its folder path changed.",

    ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS:
        "Album title is duplicated or ambiguous in at least one inventory, so title-based comparison is not reliable for this album.",

    ChangeType.FOLDER_PATH_MISSING_FROM_CURRENT:
        "Folder path exists in backup but not in the current default Photos Library. Albums or assets may still exist elsewhere.",

    ChangeType.FOLDER_PATH_NEW_IN_CURRENT:
        "Folder path exists in the current default Photos Library but not in backup. Usually normal if the folder was created later.",
}


FIELD_CHANGE_TYPES = {
    "description": ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION,
    "keywords": ChangeType.ASSET_FIELD_CHANGED_KEYWORDS,
    "favorite": ChangeType.ASSET_FIELD_CHANGED_FAVORITE,
    "hidden": ChangeType.ASSET_FIELD_CHANGED_HIDDEN,
    "date": ChangeType.ASSET_FIELD_CHANGED_DATE,
    "date_added": ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED,
    "original_filename": ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME,
    "is_movie": ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE,
}

## Photos UUID differs across Photos Libraries, so comparison uses photo_library_asset_unique_id.

def make_asset_index(inventory):
    # Build photo_library_asset_unique_id -> Asset object.
    #
    # Photos UUID is local to one Photos Library database.
    # It cannot be used to match the same asset across different
    # Photos Library currents or backups.
    index = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            raise RuntimeError(
                "Asset is missing photo_library_asset_unique_id. "
                "Run fill_photo_library_asset_unique_ids() first."
            )

        if unique_id in index:
            raise RuntimeError(
                "Duplicate photo_library_asset_unique_id found. "
                "Do not run comparison until the unique ID scheme is strengthened."
            )

        index[unique_id] = asset

    return index


def make_album_title_index(inventory):
    # Build album_title -> list[Album object].
    # Album title may not be globally unique, so keep a list.
    index = {}

    for album in inventory["albums"].values():
        title = album["title"] or ""

        if title not in index:
            index[title] = []

        index[title].append(album)

    return index


def make_folder_path_index(inventory):
    # Build folder_path -> list[Folder object].
    # Folder path may theoretically collide, so keep a list.
    index = {}

    for folder in inventory["folders"].values():
        path = folder["path"] or ""

        if path not in index:
            index[path] = []

        index[path].append(folder)

    return index


def asset_display_name(asset):
    # Prefer original filename for human reading.
    if asset is None:
        return None

    return asset["original_filename"] or asset["filename"] or asset["uuid"]


def album_folder_paths(album):
    # Return sorted folder paths for one Album object.
    return sorted(
        folder["path"]
        for folder in album["folders"].values()
    )


def asset_album_titles(asset):
    # Return sorted album titles for one Asset object.
    return sorted(
        album["title"] or ""
        for album in asset["albums"].values()
    )


def asset_folder_paths(asset):
    # Return sorted folder paths for one Asset object.
    return sorted(
        folder["path"] or ""
        for folder in asset["folders"].values()
    )


def add_diff(
    diff_records,
    change_type,
    scope,
    backup_object,
    current_object,
    backup_value,
    current_value,
    changed_field=None,
    note=None,
):
    # Add one normalized comparison record.
    if isinstance(change_type, ChangeType):
        change_type_value = change_type.value
        change_type_description = CHANGE_TYPE_DESCRIPTIONS.get(change_type)
    else:
        raise TypeError(f"change_type must be ChangeType, got: {change_type}")

    record = {
        "change_type": change_type_value,
        "change_type_description": change_type_description,
        "scope": scope,

        "changed_field": changed_field,

        "backup_object": backup_object,
        "current_object": current_object,

        "backup_value": backup_value,
        "current_value": current_value,

        "note": note,
    }

    diff_records.append(record)


def compare_asset_existence(inventory_backup, inventory_current, diff_records):
    # Compare derived asset ID existence.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    backup_ids = set(backup_assets)
    current_ids = set(current_assets)

    for asset_id in sorted(backup_ids - current_ids):
        backup_asset = backup_assets[asset_id]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_MISSING_FROM_CURRENT,
            scope="asset",
            backup_object=backup_asset,
            current_object=None,
            backup_value=asset_display_name(backup_asset),
            current_value=None,
            note="Asset exists in backup but not in the current default Photos Library. This is a high-priority possible iCloud crash data-loss case.",
        )

    for asset_id in sorted(current_ids - backup_ids):
        current_asset = current_assets[asset_id]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_NEW_IN_CURRENT,
            scope="asset",
            backup_object=None,
            current_object=current_asset,
            backup_value=None,
            current_value=asset_display_name(current_asset),
            note="Asset exists in the current default Photos Library but not in backup. This is usually normal because the current is later.",
        )


def compare_asset_metadata(inventory_backup, inventory_current, diff_records):
    # Compare metadata for assets with the same derived asset ID.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sorted(set(backup_assets) & set(current_assets))

    fields_to_compare = [
        "original_filename",
        "is_movie",
        "date",
        "date_added",
        "description",
        "keywords",
        "favorite",
        "hidden",
    ]

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        for field in fields_to_compare:
            backup_value = backup_asset.get(field)
            current_value = current_asset.get(field)

            if backup_value != current_value:
                add_diff(
                    diff_records=diff_records,
                    change_type=FIELD_CHANGE_TYPES[field],
                    scope="asset",
                    backup_object=backup_asset,
                    current_object=current_asset,
                    backup_value=backup_value,
                    current_value=current_value,
                    changed_field=field,
                    note=f"Same derived asset ID but asset field changed: {field}",
                )


def compare_asset_album_membership(inventory_backup, inventory_current, diff_records):
    # Compare album membership by derived asset ID and album title.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sorted(set(backup_assets) & set(current_assets))

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_titles = set(asset_album_titles(backup_asset))
        current_titles = set(asset_album_titles(current_asset))

        removed_titles = sorted(backup_titles - current_titles)
        added_titles = sorted(current_titles - backup_titles)

        if removed_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                current_object=current_asset,
                backup_value=removed_titles,
                current_value=None,
                note="Asset still exists, but some backup album memberships are missing from the current default Photos Library.",
            )

        if added_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                current_object=current_asset,
                backup_value=None,
                current_value=added_titles,
                note="Asset has album memberships in current that did not exist in backup. Often normal for later current.",
            )


def compare_asset_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare folder paths attached to the same asset.
    # These are derived through asset -> album_info -> folder_list.
    backup_assets = make_asset_index(inventory_backup)
    current_assets = make_asset_index(inventory_current)

    common_ids = sorted(set(backup_assets) & set(current_assets))

    for asset_id in common_ids:
        backup_asset = backup_assets[asset_id]
        current_asset = current_assets[asset_id]

        backup_paths = set(asset_folder_paths(backup_asset))
        current_paths = set(asset_folder_paths(current_asset))

        if backup_paths == current_paths:
            continue

        if backup_paths and not current_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_REMOVED
            note = "Asset still exists, but its folder paths are gone in the current default Photos Library."
        elif not backup_paths and current_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_ADDED
            note = "Asset has folder paths in the current default Photos Library but did not have them in backup."
        else:
            change_type = ChangeType.ASSET_FOLDER_PATHS_CHANGED
            note = "Asset still exists, but folder paths changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="asset_folder_paths",
            backup_object=backup_asset,
            current_object=current_asset,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note=note,
        )


def compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare albums by album title.
    # Title is the human-facing identity; UUID may not be stable across libraries.
    backup_albums_by_title = make_album_title_index(inventory_backup)
    current_albums_by_title = make_album_title_index(inventory_current)

    backup_titles = set(backup_albums_by_title)
    current_titles = set(current_albums_by_title)

    for title in sorted(backup_titles - current_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_MISSING_FROM_CURRENT,
            scope="album",
            backup_object=backup_albums_by_title[title],
            current_object=None,
            backup_value=title,
            current_value=None,
            note="Album title exists in backup but not in the current default Photos Library. This may require album reconstruction.",
        )

    for title in sorted(current_titles - backup_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_NEW_IN_CURRENT,
            scope="album",
            backup_object=None,
            current_object=current_albums_by_title[title],
            backup_value=None,
            current_value=title,
            note="Album title exists in the current default Photos Library but not in backup. Usually normal for later current.",
        )

    for title in sorted(backup_titles & current_titles):
        backup_albums = backup_albums_by_title[title]
        current_albums = current_albums_by_title[title]

        if len(backup_albums) != 1 or len(current_albums) != 1:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS,
                scope="album",
                backup_object=backup_albums,
                current_object=current_albums,
                backup_value=len(backup_albums),
                current_value=len(current_albums),
                note="Album title is not unique in at least one inventory. Folder comparison by title is ambiguous.",
            )
            continue

        backup_album = backup_albums[0]
        current_album = current_albums[0]

        backup_paths = set(album_folder_paths(backup_album))
        current_paths = set(album_folder_paths(current_album))

        if backup_paths == current_paths:
            continue

        if backup_paths and not current_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_REMOVED
            note = "Album still exists, but it is no longer inside any folder path in the current default Photos Library."
        elif not backup_paths and current_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_ADDED
            note = "Album gained folder paths in the current default Photos Library."
        else:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_CHANGED
            note = "Album still exists, but its folder path changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="album_folder_paths",
            backup_object=backup_album,
            current_object=current_album,
            backup_value=sorted(backup_paths),
            current_value=sorted(current_paths),
            note=note,
        )


def compare_folder_paths(inventory_backup, inventory_current, diff_records):
    # Compare folder paths by human-readable path.
    backup_folders_by_path = make_folder_path_index(inventory_backup)
    current_folders_by_path = make_folder_path_index(inventory_current)

    backup_paths = set(backup_folders_by_path)
    current_paths = set(current_folders_by_path)

    for path in sorted(backup_paths - current_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_MISSING_FROM_CURRENT,
            scope="folder",
            backup_object=backup_folders_by_path[path],
            current_object=None,
            backup_value=path,
            current_value=None,
            note="Folder path exists in backup but not in the current default Photos Library. Albums/assets may still exist elsewhere.",
        )

    for path in sorted(current_paths - backup_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_NEW_IN_CURRENT,
            scope="folder",
            backup_object=None,
            current_object=current_folders_by_path[path],
            backup_value=None,
            current_value=path,
            note="Folder path exists in the current default Photos Library but not in backup.",
        )


def compare_inventories(inventory_backup, inventory_current):
    # Run all comparison passes and return normalized diff records.
    diff_records = []

    compare_asset_existence(inventory_backup, inventory_current, diff_records)
    compare_asset_metadata(inventory_backup, inventory_current, diff_records)
    compare_asset_album_membership(inventory_backup, inventory_current, diff_records)
    compare_asset_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_album_existence_and_folder_paths(inventory_backup, inventory_current, diff_records)
    compare_folder_paths(inventory_backup, inventory_current, diff_records)

    return diff_records


def summarize_diff_records(diff_records):
    # Count diff records by change_type.
    summary = {}

    for record in diff_records:
        change_type = record["change_type"]

        if change_type not in summary:
            summary[change_type] = 0

        summary[change_type] += 1

    return dict(sorted(summary.items()))
